In [3]:
# Jupyter cell — VS Code

from pathlib import Path
import re
import pandas as pd

# ---------- Paths ----------
SAMPLE_CSV = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\2_Stratified_Sampling\Final\stratified_sample_moe10.csv")
OUT_XLSX   = SAMPLE_CSV.with_name("stratified_sample_moe10_output.xlsx")

CONFIG_DIR = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ1\All_Config_Files")
TESTS_DIR  = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ1\All_Test_Files")

# ---------- Helpers ----------
YAML_EXTS = {".yml", ".yaml"}
GRADLE_EXTS = {".gradle"}  # .gradle.kts handled via name.endswith

EXECENV_ALLOWED = [
    "Emulator_ReactiveCircus",
    "Emulator_Malinskiy",
    "Emulator_DIY",
    "Emulator_Other",
    "GMD_intent",
    "Emulator_GMD",
    "Third Party",
    "Real Device",
    "Unknown",
]
INVOC_ALLOWED = ["Gradle", "ADB", "3P CLIs", "Unknown"]
CONF_ALLOWED  = ["high", "medium", ""]

def truth01(val):
    s = pd.Series([val])
    if s.dtype == bool:
        return int(s.iloc[0])
    t = s.astype(str).str.strip().str.lower()
    return int(t.iloc[0] in {"1","true","t","yes","y","on"})

def norm_cat(x: str, allowed: list) -> str:
    if pd.isna(x):
        return ""
    s = str(x).strip()
    low = s.lower()
    aliases = {
        "reactivecircus": "Emulator_ReactiveCircus",
        "malinskiy": "Emulator_Malinskiy",
        "diy": "Emulator_DIY",
        "other": "Emulator_Other",
        "third party": "Third Party",
        "third_party": "Third Party",
        "third-party": "Third Party",
        "real": "Real Device",
        "real device": "Real Device",
        "gmd": "Emulator_GMD",
        "gmd_intent": "GMD_intent",
        "3p": "3P CLIs",
        "3p clis": "3P CLIs",
        "third party clis": "3P CLIs",
        "hi": "high",
        "med": "medium",
    }
    if low in aliases:
        s = aliases[low]
    # exact or case-insensitive match
    if s in allowed: return s
    for a in allowed:
        if s.lower() == a.lower():
            return a
    return s

def safe_read_text(p: Path) -> str:
    for enc in ("utf-8", "latin-1"):
        try:
            return p.read_text(encoding=enc, errors="ignore")
        except Exception:
            pass
    return ""

def repo_key_from_filename(fname: str) -> str:
    base = Path(fname).name
    key = base.split("__", 1)[0] if "__" in base else Path(base).stem
    return key.strip().lower()

def index_repo_files(root: Path):
    idx = {}
    if root.exists():
        for p in root.rglob("*"):
            if not p.is_file():
                continue
            is_yaml   = p.suffix.lower() in YAML_EXTS
            is_gradle = (p.suffix.lower() in GRADLE_EXTS) or p.name.lower().endswith(".gradle.kts")
            if is_yaml or is_gradle:
                key = repo_key_from_filename(p.name)
                idx.setdefault(key, []).append(p)
    return idx

def build_androidtest_presence_index(tests_dir: Path):
    present = set()
    if tests_dir.exists():
        for p in tests_dir.iterdir():
            if p.is_file():
                present.add(repo_key_from_filename(p.name))
    return present

def any_hit(text: str, patterns: dict) -> bool:
    return any(re.search(p, text, flags=re.IGNORECASE | re.DOTALL | re.MULTILINE) for p in patterns.values())

def find_any(text: str, pattern_list) -> bool:
    return any(re.search(p, text, flags=re.IGNORECASE | re.DOTALL | re.MULTILINE) for p in pattern_list)

def search_one(text: str, pattern: str):
    m = re.search(pattern, text, flags=re.IGNORECASE | re.DOTALL | re.MULTILINE)
    return bool(m), (m.group(0)[:200] if m else "")

def looks_like_real_device_serial(cmd: str) -> bool:
    """
    Heuristic: adb -s SERIAL ... where SERIAL does not look like an emulator-5554/localhost:5555.
    """
    m = re.search(r"\badb\s+-s\s+([^\s]+)", cmd, flags=re.IGNORECASE)
    if not m:
        return False
    serial = m.group(1).lower()
    if serial.startswith("emulator-") or serial.startswith("localhost:"):
        return False
    return True

# --- Patterns aligned with detection taxonomy ---
YAML_SIGNAL_PATTERNS = {
    # Popular GH Actions
    "reactivecircus_runner": r"uses:\s*reactivecircus/android-emulator-runner@",
    # malinskiy variations
    "malinskiy_runner_a": r"uses:\s*malinskiy/android-emulator-runner@",
    "malinskiy_runner_b": r"uses:\s*malinskiy/action-android(?:@|/)",
    # Gradle-based IT triggers
    "gradle_connected": r"\b(?:^|[^\w])gradle[w]?\b[^\n\r]*\bconnected(?:androidtest|check)\b",
    "gradle_managed_tasks": r"\bmanagedDevice\w*AndroidTest\b|\b(alldevicechecks|devicecheck)s?\b",
    "variant_androidtest": r"\b[a-zA-Z0-9_]+(?:Debug|Release)?AndroidTest\b",
    "baseline_profile": r"\b(generate|collect)BaselineProfile\b",
    # ADB instrumentation
    "adb_instrument": r"\badb\s+(?:-s\s+\S+\s+)?shell\s+am\s+instrument\b",
    # Third-party CLIs
    "gcloud_ftl": r"\bgcloud\b[^\n\r]*\bfirebase\s+test\s+android\s+run\b",
    "flank": r"\bflank\b[^\n\r]*\bandroid\b",
    "saucectl": r"\bsaucectl\b",
    "appcenter": r"\bappcenter\s+test\s+run\s+android\b",
    "maestro_cloud": r"\bmaestro\s+cloud\b",
    "emulator_wtf": r"\bemulator\.wtf\b|\bew-cli\b",
    "browserstack": r"\bbrowserstack\b|\bbstack\b",
    "saucelabs": r"\bsauce(labs)?\b",
    # Emulator DIY hints
    "emulator_launch": r"\bemulator\b[^\n\r]*(?:-avd\s+\S+|@\S+)",
    "avdmanager": r"\bavdmanager\b|\bandroid\s+create\s+avd\b",
    "sdk_sysimg": r"\bsdkmanager\b[^\n\r]*system-images;android-",
    # Flutter integration test signal
    "flutter_android": r"\bflutter\s+(?:drive|test)\b[^\n\r]*(integration_test|/integration_test/)",
    "dart_integ": r"\bdart\s+test\b[^\n\r]*(integration_test|/integration_test/)",
    # GMD intent in YAML
    "gmd_yaml": r"\b(ManagedVirtualDevice|managedDevices|deviceGroups)\b",
}

BUILD_SIGNAL_PATTERNS = {
    "testInstrumentationRunner": r"\btestInstrumentationRunner\b\s*(=|\s)\s*['\"][^'\"]+['\"]?",
    "androidTestDependency_str": r"\bandroidTest(?:Implementation|Api|CompileOnly|RuntimeOnly)\s*\(?\s*['\"][^'\"]+['\"]",
    "androidTestDependency_alias": r"\bandroidTest(?:Implementation|Api|CompileOnly|RuntimeOnly)\s*\(\s*[a-zA-Z0-9_.:-]+\s*\)",
    "androidx_test": r"['\"][^'\"]*androidx\.test[^'\"]*['\"]",
    "espresso": r"['\"][^'\"]*espresso[^'\"]*['\"]",
    "uiautomator": r"['\"][^'\"]*uiautomator[^'\"]*['\"]",
    "orchestrator": r"['\"][^'\"]*androidx\.test:orchestrator[^'\"]*['\"]",
    "benchmark": r"['\"][^'\"]*androidx\.benchmark[^'\"]*['\"]",
    "useOrchestrator_true": r"\buseOrchestrator\b\s*(=|\s)\s*true\b",
    "connectedAndroidTest": r"\bconnectedAndroidTest\b",
    # GMD confirmed in build/configs:
    "gmd_block_hint": r"\btestOptions\s*\{[^}]*managedDevices\b|testOptions\.managedDevices|ManagedVirtualDevice",
}

# ---------- Load sample & build indices ----------
df = pd.read_csv(SAMPLE_CSV, low_memory=False)
if "full_name" not in df.columns:
    raise KeyError("Sample must contain a 'full_name' column.")

cfg_index = index_repo_files(CONFIG_DIR)
at_present = build_androidtest_presence_index(TESTS_DIR)

# ---------- Detection per repo key ----------
keys = df["full_name"].astype(str).str.strip().str.lower()
uniq = keys.unique()

scan_result = {}
for k in uniq:
    yaml_found = 0
    build_found = 0
    texts_yaml, texts_gradle = [], []
    # aggregate text by type
    for p in cfg_index.get(k, []):
        t = safe_read_text(p)
        if not t:
            continue
        is_yaml   = p.suffix.lower() in YAML_EXTS
        is_gradle = (p.suffix.lower() in GRADLE_EXTS) or p.name.lower().endswith(".gradle.kts")
        if is_yaml:
            texts_yaml.append(t)
            if any_hit(t, YAML_SIGNAL_PATTERNS):
                yaml_found = 1
        if is_gradle:
            texts_gradle.append(t)
            if any_hit(t, BUILD_SIGNAL_PATTERNS):
                build_found = 1

    yaml_text = "\n".join(texts_yaml)
    gradle_text = "\n".join(texts_gradle)

    # --- fine-grained flags from YAML/Gradle ---
    # Exec env signals
    rc = bool(re.search(YAML_SIGNAL_PATTERNS["reactivecircus_runner"], yaml_text, re.I|re.S|re.M))
    ml = (bool(re.search(YAML_SIGNAL_PATTERNS["malinskiy_runner_a"], yaml_text, re.I|re.S|re.M)) or
          bool(re.search(YAML_SIGNAL_PATTERNS["malinskiy_runner_b"], yaml_text, re.I|re.S|re.M)))
    diy = find_any(yaml_text, [YAML_SIGNAL_PATTERNS["emulator_launch"], YAML_SIGNAL_PATTERNS["avdmanager"], YAML_SIGNAL_PATTERNS["sdk_sysimg"]])
    gmd_yaml_intent = bool(re.search(YAML_SIGNAL_PATTERNS["gmd_yaml"], yaml_text, re.I|re.S|re.M))
    gmd_build = bool(re.search(BUILD_SIGNAL_PATTERNS["gmd_block_hint"], gradle_text, re.I|re.S|re.M))

    # Third party signals
    third_party = find_any(
        yaml_text,
        [
            YAML_SIGNAL_PATTERNS["gcloud_ftl"], YAML_SIGNAL_PATTERNS["flank"],
            YAML_SIGNAL_PATTERNS["saucectl"], YAML_SIGNAL_PATTERNS["appcenter"],
            YAML_SIGNAL_PATTERNS["maestro_cloud"], YAML_SIGNAL_PATTERNS["emulator_wtf"],
            YAML_SIGNAL_PATTERNS["browserstack"], YAML_SIGNAL_PATTERNS["saucelabs"]
        ]
    )

    # Real device vs emulator via adb -s
    real_device = False
    for m in re.finditer(r"\badb\s+-s\s+[^\s]+[^\n\r]*", yaml_text, flags=re.I):
        if looks_like_real_device_serial(m.group(0)):
            real_device = True
            break

    # Invocation
    inv_gradle = find_any(yaml_text, [
        YAML_SIGNAL_PATTERNS["gradle_connected"],
        YAML_SIGNAL_PATTERNS["gradle_managed_tasks"],
        YAML_SIGNAL_PATTERNS["variant_androidtest"],
        YAML_SIGNAL_PATTERNS["baseline_profile"],
    ]) or bool(re.search(BUILD_SIGNAL_PATTERNS["connectedAndroidTest"], gradle_text, re.I|re.S|re.M))
    inv_adb = bool(re.search(YAML_SIGNAL_PATTERNS["adb_instrument"], yaml_text, re.I|re.S|re.M))
    inv_3p  = third_party

    # Flutter IT
    flutter_it = find_any(yaml_text, [YAML_SIGNAL_PATTERNS["flutter_android"], YAML_SIGNAL_PATTERNS["dart_integ"]])

    # Build-only instrumentation hints (deps/runner/orchestrator)
    build_it_deps = any_hit(gradle_text, BUILD_SIGNAL_PATTERNS)

    # AT presence (file-level)
    at_found = 1 if k in at_present else 0

    # --- Decide ExecEnv ---
    if third_party:
        exec_env = "Third Party"
    elif real_device:
        exec_env = "Real Device"
    elif rc:
        exec_env = "Emulator_ReactiveCircus"
    elif ml:
        exec_env = "Emulator_Malinskiy"
    elif gmd_build:
        exec_env = "Emulator_GMD"
    elif gmd_yaml_intent:
        exec_env = "GMD_intent"
    elif diy:
        exec_env = "Emulator_DIY"
    elif inv_gradle and (rc or ml or diy or gmd_yaml_intent or gmd_build):
        exec_env = "Emulator_Other"
    else:
        exec_env = "Unknown"

    # --- Decide Invocation ---
    if inv_3p:
        invocation = "3P CLIs"
    elif inv_adb:
        invocation = "ADB"
    elif inv_gradle:
        invocation = "Gradle"
    else:
        invocation = "Unknown"

    # --- CI signal (boolean) ---
    ci_signal = int(any([
        inv_gradle, inv_adb, inv_3p,
        build_it_deps,
        at_found == 1
    ]))

    # --- Confidence (heuristic, optional) ---
    # high if both env and invocation determined (not Unknown),
    # medium if exactly one is determined, else blank
    det_env = exec_env != "Unknown"
    det_inv = invocation != "Unknown"
    if det_env and det_inv:
        confidence = "high"
    elif det_env or det_inv:
        confidence = "medium"
    else:
        confidence = ""

    scan_result[k] = {
        # legacy booleans
        "YAML_Check": int(yaml_found),
        "Build_Check": int(build_found),
        "AT_Check": int(at_found),
        # new fields
        "ExecEnv_Check": exec_env,
        "Invocation_Check": invocation,
        "CI_Signal_Check": int(ci_signal),
        "Flutter_IT_Check": int(flutter_it),
        "Confidence_Check": confidence,
    }

# ---------- Map back to rows ----------
df["YAML_Check"]       = [scan_result.get(k, {}).get("YAML_Check", 0) for k in keys]
df["Build_Check"]      = [scan_result.get(k, {}).get("Build_Check", 0) for k in keys]
df["AT_Check"]         = [scan_result.get(k, {}).get("AT_Check", 0) for k in keys]
df["ExecEnv_Check"]    = [scan_result.get(k, {}).get("ExecEnv_Check", "Unknown") for k in keys]
df["Invocation_Check"] = [scan_result.get(k, {}).get("Invocation_Check", "Unknown") for k in keys]
df["CI_Signal_Check"]  = [scan_result.get(k, {}).get("CI_Signal_Check", 0) for k in keys]
df["Flutter_IT_Check"] = [scan_result.get(k, {}).get("Flutter_IT_Check", 0) for k in keys]
df["Confidence_Check"] = [scan_result.get(k, {}).get("Confidence_Check", "") for k in keys]

# ---------- Mismatch Note vs any *_pred columns ----------
def add_mismatch(mismatches, name, pred, check, cat_allowed=None):
    if pd.isna(pred):
        return
    if cat_allowed is None:
        # boolean compare
        yp = truth01(pred)
        yc = truth01(check)
        if yp != yc:
            mismatches.append(f"{name} mismatch (pred={yp}, found={yc})")
    else:
        yp = norm_cat(pred, cat_allowed)
        yc = norm_cat(check, cat_allowed)
        if yp != yc:
            mismatches.append(f"{name} mismatch (pred='{yp}', found='{yc}')")

notes = []
for i, row in df.iterrows():
    mm = []
    # legacy
    if "YAML_pred"  in df.columns: add_mismatch(mm, "YAML",  row["YAML_pred"],  row["YAML_Check"])
    if "Build_pred" in df.columns: add_mismatch(mm, "Build", row["Build_pred"], row["Build_Check"])
    if "AT_pred"    in df.columns: add_mismatch(mm, "AT",    row["AT_pred"],    row["AT_Check"])
    # new
    if "ExecEnv_pred"    in df.columns: add_mismatch(mm, "ExecEnv",    row["ExecEnv_pred"],    row["ExecEnv_Check"],    EXECENV_ALLOWED)
    if "Invocation_pred" in df.columns: add_mismatch(mm, "Invocation", row["Invocation_pred"], row["Invocation_Check"], INVOC_ALLOWED)
    if "CI_Signal_pred"  in df.columns: add_mismatch(mm, "CI_Signal",  row["CI_Signal_pred"],  row["CI_Signal_Check"])
    if "Flutter_IT_pred" in df.columns: add_mismatch(mm, "Flutter_IT", row["Flutter_IT_pred"], row["Flutter_IT_Check"])
    if "Confidence_pred" in df.columns: add_mismatch(mm, "Confidence", row["Confidence_pred"], row["Confidence_Check"], CONF_ALLOWED)
    notes.append("; ".join(mm))

df["Note"] = notes

# ---------- Save Excel next to input (original cols + new check cols) ----------
# Keep original order, append new columns if not already present
new_cols_order = [
    "YAML_Check","Build_Check","AT_Check",
    "CI_Signal_Check","ExecEnv_Check","Invocation_Check","Flutter_IT_Check","Confidence_Check",
    "Note"
]
for c in new_cols_order:
    if c not in df.columns:
        df[c] = ""

df.to_excel(OUT_XLSX, index=False)
print(f"[DONE] Wrote: {OUT_XLSX}")


[DONE] Wrote: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\2_Stratified_Sampling\Final\stratified_sample_moe10_output.xlsx


## accuracy check

In [4]:
# %% [markdown]
# Per-stratum accuracy + per-factor PR/F1 + overall (micro/macro) PR/F1
# Input:  C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\2_Stratified_Sampling\Final\stratified_sample_moe10_output.xlsx
# Output: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\2_Stratified_Sampling\Final\validation_metrics.xlsx

# %%
import pandas as pd
import numpy as np
from pathlib import Path

# ---------- CONFIG ----------
FOLDER = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\2_Stratified_Sampling\Final")
IN_XLSX = FOLDER / "stratified_sample_moe10_output.xlsx"
OUT_XLSX = FOLDER / "validation_metrics.xlsx"

# ---------- Helpers ----------
def to01(s: pd.Series) -> pd.Series:
    """Robust boolean -> {0,1}."""
    if s.dtype == bool:
        return s.astype(int)
    if pd.api.types.is_numeric_dtype(s):
        return (pd.to_numeric(s, errors="coerce").fillna(0) != 0).astype(int)
    t = s.astype(str).str.strip().str.lower()
    truthy = {"1","true","t","yes","y","on"}
    falsy  = {"0","false","f","no","n","off","","none","null","nan"}
    out = pd.Series(np.nan, index=s.index, dtype="float")
    out[t.isin(truthy)] = 1
    out[t.isin(falsy)]  = 0
    # fallback numeric
    num = pd.to_numeric(t.str.replace(r"[^0-9\.\-]+","", regex=True), errors="coerce")
    out = out.where(out.notna(), (num.fillna(0) != 0).astype(int))
    return out.astype(int)

def pr_from_counts(tp, fp, fn):
    prec = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    rec  = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    f1   = (2*prec*rec)/(prec+rec) if (pd.notna(prec) and pd.notna(rec) and (prec+rec)>0) else np.nan
    return prec, rec, f1

def counts_for(pred, true):
    tp = int(((pred==1) & (true==1)).sum())
    fp = int(((pred==1) & (true==0)).sum())
    fn = int(((pred==0) & (true==1)).sum())
    tn = int(((pred==0) & (true==0)).sum())
    return tp, fp, fn, tn

# ---------- Load ----------
df = pd.read_excel(IN_XLSX)

need = ["YAML_pred","Build_pred","AT_pred","YAML_Check","Build_Check","AT_Check"]
missing = [c for c in need if c not in df.columns]
if missing:
    raise KeyError(f"Missing column(s) in input: {missing}")

# Normalize to 0/1
Yp = to01(df["YAML_pred"]);  Bp = to01(df["Build_pred"]);  Ap = to01(df["AT_pred"])
Yt = to01(df["YAML_Check"]); Bt = to01(df["Build_Check"]); At = to01(df["AT_Check"])

# Ensure stratum label (from predictions if not provided)
if "stratum" in df.columns:
    strata = df["stratum"].astype(str)
else:
    strata = pd.Series([f"Y{y}_B{b}_A{a}" for y,b,a in zip(Yp,Bp,Ap)], index=df.index)

# ---------- 1) Per-stratum accuracy (exact triplet match) ----------
triplet_match = (Yp.eq(Yt) & Bp.eq(Bt) & Ap.eq(At)).astype(int)
per_stratum = (
    pd.DataFrame({"stratum": strata, "match": triplet_match})
      .groupby("stratum", dropna=False)
      .agg(n=("match","size"), matches=("match","sum"))
      .reset_index()
      .sort_values("stratum", ignore_index=True)
)
per_stratum["accuracy"] = per_stratum["matches"] / per_stratum["n"]
overall_triplet_accuracy = float(triplet_match.mean())

# ---------- 2) Per-factor precision/recall/F1 (whole sample) ----------
rows = []
for label, pred, true in [("YAML",Yp,Yt), ("Build",Bp,Bt), ("AT",Ap,At)]:
    tp, fp, fn, tn = counts_for(pred, true)
    prec, rec, f1 = pr_from_counts(tp, fp, fn)
    rows.append({
        "factor": label,
        "N": int(len(pred)),
        "TP": tp, "FP": fp, "FN": fn, "TN": tn,
        "precision": round(prec,4) if pd.notna(prec) else np.nan,
        "recall":    round(rec, 4) if pd.notna(rec) else np.nan,
        "f1":        round(f1,   4) if pd.notna(f1) else np.nan,
    })
per_factor_metrics = pd.DataFrame(rows)

# ---------- 3) OVERALL precision/recall/F1 (whole sample) ----------
# Micro-average (pool all Y,B,AT decisions):
tp = per_factor_metrics["TP"].sum()
fp = per_factor_metrics["FP"].sum()
fn = per_factor_metrics["FN"].sum()
prec_micro, rec_micro, f1_micro = pr_from_counts(tp, fp, fn)

# Macro-average (unweighted mean across factors):
prec_macro = per_factor_metrics["precision"].mean(skipna=True)
rec_macro  = per_factor_metrics["recall"].mean(skipna=True)
f1_macro   = per_factor_metrics["f1"].mean(skipna=True)

overall_metrics = pd.DataFrame([
    {"overall_type":"micro", "precision":round(prec_micro,4), "recall":round(rec_micro,4), "f1":round(f1_micro,4)},
    {"overall_type":"macro", "precision":round(prec_macro,4),  "recall":round(rec_macro,4),  "f1":round(f1_macro,4)},
    {"overall_type":"triplet_exact_accuracy", "precision":np.nan, "recall":np.nan, "f1":np.nan}
])
# Attach the triplet exact-match accuracy as a side note:
overall_note = pd.DataFrame([{"overall_triplet_exact_accuracy": round(overall_triplet_accuracy,4),
                              "rows": len(df)}])

# ---------- Save ----------
with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as xw:
    per_stratum.to_excel(xw, sheet_name="Stratum_Accuracy", index=False)
    per_factor_metrics.to_excel(xw, sheet_name="Per_Factor_PRF1", index=False)
    overall_metrics.to_excel(xw, sheet_name="Overall_PRF1", index=False)
    overall_note.to_excel(xw, sheet_name="Overall_Note", index=False)

print(f"[OK] Saved metrics → {OUT_XLSX}")
print("Overall (micro) P/R/F1:", round(prec_micro,4), round(rec_micro,4), round(f1_micro,4))
print("Overall (macro) P/R/F1:", round(prec_macro,4), round(rec_macro,4), round(f1_macro,4))
print("Triplet exact-match accuracy:", round(overall_triplet_accuracy,4))


[OK] Saved metrics → C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\2_Stratified_Sampling\Final\validation_metrics.xlsx
Overall (micro) P/R/F1: 0.9235 0.9848 0.9532
Overall (macro) P/R/F1: 0.9045 0.9818 0.9374
Triplet exact-match accuracy: 0.8724
